# Upload XGBoost Multiclass Model to Hugging Face

This notebook uploads the final **XGBoost multiclass ATSEG classifier** to Hugging Face.

The model predicts one of three classes:

- `SEG_A`
- `SEG_B`
- `SEG_C`

This notebook assumes that the trained model artifacts already exist locally. It does **not** retrain the model.

## 1. Environment setup

This section checks which Python environment is being used and imports the required libraries. If `huggingface_hub` is not installed in the current notebook kernel, install it using the same Python executable shown below.

In [7]:
import sys

print(sys.executable)

/usr/local/bin/python3.11


## 2. Hugging Face login

Run this cell once per environment/session. It will ask for a Hugging Face access token.

The token must have **write** permissions in order to create or update a repository.

In [ ]:
#Login to Hugging Face Hub
from huggingface_hub import login

login()

## 3. Define local artifact folder

Set `UPLOAD_DIR` to the folder that contains the model artifacts to be uploaded.

Expected files:

- `best_xgb_multiclass_candidate58.joblib`
- `xgb_multiclass_candidate58_metadata.json`
- `xgb_candidate58_test_predictions.csv`
- `xgb_multiclass_hyperparameter_search_results.csv`

In [ ]:
# List files in the upload directory
from pathlib import Path
import json

UPLOAD_DIR = Path("/Users/omarleosamman/Downloads/hf_xgb_candidate58_upload")

print("Files in upload folder:")
for file in UPLOAD_DIR.iterdir():
    print("-", file.name)

Files in upload folder:
- xgb_multiclass_candidate58_test_predictions_with_hcp_id.csv
- xgb_multiclass_test_predictions.csv
- xgb_multiclass_candidate58_test_predictions.csv
- best_xgb_multiclass_atseg_candidate58.joblib
- xgb_multiclass_candidate58_metadata.json


## 4. Load model metadata

The metadata file stores the model context required for reuse:

- model type
- selected candidate
- best hyperparameters
- label mapping
- test metrics


In [ ]:
# Load and display the metadata
metadata_file = UPLOAD_DIR / "xgb_multiclass_candidate58_metadata.json"

with open(metadata_file, "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata

{'model': 'XGBClassifier',
 'task': 'SEG_A vs SEG_B vs SEG_C',
 'selection_reason': 'Candidate 58 selected due to better SEG_C recall while maintaining competitive Macro F1.',
 'best_params': {'subsample': 0.9,
  'reg_lambda': 1,
  'reg_alpha': 0.1,
  'n_estimators': 200,
  'min_child_weight': 1,
  'max_depth': 3,
  'learning_rate': 0.03,
  'gamma': 0.05,
  'colsample_bytree': 0.75},
 'decision_rule': 'argmax over predicted class probabilities',
 'label_mapping': {'0': 'SEG_A', '1': 'SEG_B', '2': 'SEG_C'},
 'test_metrics': {'accuracy': 0.6449579831932774,
  'macro_f1': 0.5792501739473731,
  'weighted_f1': 0.6479920763036838,
  'SEG_A_precision': 0.7828525641025641,
  'SEG_A_recall': 0.7626854020296643,
  'SEG_A_f1': 0.7726374060893634,
  'SEG_B_precision': 0.5787878787878787,
  'SEG_B_recall': 0.5701492537313433,
  'SEG_B_f1': 0.5744360902255639,
  'SEG_C_precision': 0.3728813559322034,
  'SEG_C_recall': 0.41025641025641024,
  'SEG_C_f1': 0.390677025527192,
  'bc_recall_avg': 0.4902028

## 5. Create the Hugging Face model card

The `README.md` file acts as the model card in Hugging Face. It explains what the model does, what files are included, the test metrics, and how to reuse the model.

In [ ]:
# Create README.md with model information and test metrics
test_metrics = metadata.get("test_metrics", {})
best_params = metadata.get("best_params", {})
model_name = metadata.get("model", "XGBClassifier")
selected_candidate = metadata.get("selected_candidate", "58")

readme_text = f'''
---
license: other
tags:
- multiclass-classification
- healthcare
- physician-segmentation
- xgboost
---

# XGBoost Multiclass ATSEG Classifier - Candidate {selected_candidate}

This repository contains the selected XGBoost multiclass model for direct ATSEG classification.

## Task

Multiclass classification:

- `0`: SEG_A
- `1`: SEG_B
- `2`: SEG_C

## Selected Model

Model: `{model_name}`  
Selected candidate: `{selected_candidate}`

Candidate {selected_candidate} was selected because it improved SEG_C recall while maintaining competitive Macro F1.

## Best Parameters

{best_params}

## Test Metrics

- Accuracy: {test_metrics.get("accuracy", "NA")}
- Macro F1: {test_metrics.get("macro_f1", "NA")}
- Weighted F1: {test_metrics.get("weighted_f1", "NA")}
- SEG_A Recall: {test_metrics.get("SEG_A_recall", "NA")}
- SEG_B Recall: {test_metrics.get("SEG_B_recall", "NA")}
- SEG_C Recall: {test_metrics.get("SEG_C_recall", "NA")}

## Files

- `best_xgb_multiclass_candidate58.joblib`: trained XGBoost multiclass model
- `xgb_multiclass_candidate58_metadata.json`: model parameters, label mapping and metrics
- `xgb_candidate58_test_predictions.csv`: test predictions with predicted probabilities
- `xgb_multiclass_hyperparameter_search_results.csv`: hyperparameter search results

## Reuse

    import joblib
    import json
    import numpy as np

    model = joblib.load("best_xgb_multiclass_candidate58.joblib")

    with open("xgb_multiclass_candidate58_metadata.json", "r") as f:
        metadata = json.load(f)

    proba = model.predict_proba(X_new_flat)
    pred_encoded = np.argmax(proba, axis=1)

    label_mapping = metadata["label_mapping"]
    pred_label = [label_mapping[str(i)] for i in pred_encoded]

## Input Format

The model expects flattened temporal tensor features with the same feature order used during training.
'''

with open(UPLOAD_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README created:", UPLOAD_DIR / "README.md")

README created: /Users/omarleosamman/Downloads/hf_xgb_candidate58_upload/README.md


## 6. Upload model artifacts to Hugging Face

This section creates or updates a Hugging Face model repository and uploads the full contents of `UPLOAD_DIR`.

In [ ]:
# Upload to Hugging Face Hub
from huggingface_hub import create_repo, upload_folder

HF_USERNAME = "omarleosamman"  # cambia esto
REPO_NAME = "xgb-multiclass-atseg-candidate58"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True
)

upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(UPLOAD_DIR),
    commit_message="Upload selected XGBoost multiclass candidate 58"
)

print(f"Uploaded to: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to: https://huggingface.co/omarleosamman/xgb-multiclass-atseg-candidate58


## 7. Final notes

The repository now contains:

- the trained XGBoost model
- metadata with label mapping and evaluation metrics
- test prediction outputs
- hyperparameter search results
- a Hugging Face model card

This allows the model to be reused without retraining, as long as new input data is prepared with the same flattened tensor format used during training.